# Test the way the world files are read

In [10]:
import os

path = "/root/e2e_crossq/src/the-barn-challenge-CrossQ/jackal_helper/worlds/BARN"
os.listdir(path)

PermissionError: [Errno 13] Permission denied: '/root/e2e_crossq/src/the-barn-challenge-CrossQ/jackal_helper/worlds/BARN'

# Test the encoders and all ML part

In [ ]:
import torch 
from sac.net import TCNEncoder, MLP_CrossQ
from gym.spaces import Box
import numpy as np


obs_dim = 720 + 2 + 2 + 2  # 720 dim laser scan + goal position + action taken in this time step 
observation_space = Box(
    low=0,
    high=4,
    shape=(obs_dim,),
    dtype=np.float32
)


print(np.concatenate((observation_space.sample(), np.array([1, 2]))).shape)

state = torch.ones([2, 4, 724])
input_dim = 744
net = TCNEncoder((4, 726), 2, 512)

min_v=-1
max_v=2
min_w=-3.14
max_w=3.14

range_dict = RANGE_DICT = {
            "linear_velocity": [min_v, max_v],
            "angular_velocity": [min_w, max_w],
        }

action_space = Box(
            low=np.array([RANGE_DICT["linear_velocity"][0], RANGE_DICT["angular_velocity"][0]]),
            high=np.array([RANGE_DICT["linear_velocity"][1], RANGE_DICT["angular_velocity"][1]]),
            dtype=np.float32
        )

actions = torch.Tensor([[action_space.sample()] for i in range(2)]).squeeze(1)

cutoff = (input_dim - 720) // 4
print("cutoff: ", cutoff)
no_laser_data = state[:, :, -cutoff:].reshape(state.shape[0], -1)
state = state[:, :, :-cutoff]
#s = state_preprocess(state) if state_preprocess else state
s = net(state)
print('State: ', s.shape)
print('No laser data: ', no_laser_data.shape)
s = torch.cat([s, no_laser_data], dim=1)
print('State: ', s.shape)
sa1 = torch.cat([s, actions], dim=1)
head = MLP_CrossQ(input_dim, 512, 2)
print("sa1: ", sa1.shape)
q1 = head(s)


Observation space:  (726,)
(728,)
cutoff:  6
State:  torch.Size([2, 718])
No laser data:  torch.Size([2, 24])
State:  torch.Size([2, 742])
sa1:  torch.Size([2, 744])


/home/bbruno/anaconda3/envs/rl_env/lib/python3.13/site-packages/gym/spaces/box.py:127: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")


RuntimeError: The size of tensor a (742) must match the size of tensor b (744) at non-singleton dimension 0

In [1]:
#from envs.multi_reward import MultiRewardEnv
import gym
from gym.spaces import Box
import numpy as np
import torch

#env = gym.make('MultiRewardEnv-v0')

min_v=-1
max_v=2
min_w=-3.14
max_w=3.14

range_dict = RANGE_DICT = {
            "linear_velocity": [min_v, max_v],
            "angular_velocity": [min_w, max_w],
        }

action_space = Box(
            low=np.array([RANGE_DICT["linear_velocity"][0], RANGE_DICT["angular_velocity"][0]]),
            high=np.array([RANGE_DICT["linear_velocity"][1], RANGE_DICT["angular_velocity"][1]]),
            dtype=np.float32
        )

actions = torch.Tensor([[action_space.sample()] for i in range(2)]).squeeze(1)
print(actions)
print(actions.shape)

#final_input = torch.cat([s, actions], dim=1)

tensor([[ 1.2852,  2.0707],
        [-0.5884, -0.4520]])
torch.Size([2, 2])


/home/gabrielga/anaconda3/envs/rl_env/lib/python3.13/site-packages/gym/spaces/box.py:127: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")
/tmp/ipykernel_18497/773663606.py:25: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  actions = torch.Tensor([[action_space.sample()] for i in range(2)]).squeeze(1)


In [2]:
from sac.net import TCNEncoder, MLP_CrossQ
from sac.rl import CrossQCritic, Actor
import torch
import numpy as np
from torch.nn import functional as F

state = torch.ones([2, 4, 366])
action = torch.Tensor([[ 1.0215,  1.5293], [-0.8067,  0.3733]])

action_dim = np.prod(action_space.shape)
action_space_low = action_space.low
action_space_high = action_space.high

laser_dim = 360

input_dim = laser_dim + 4*(2 + 2 + 2) # Input dim is [laser dimensio + stack frames*(size of local goal + action dim + size of global goal)]
actor = Actor(
    state_preprocess=TCNEncoder((4, laser_dim), 2, 512, use_continual_backprop = True, batch_norm = True),
    head=MLP_CrossQ(
        input_dim,
        2,
        512,
        use_continual_backprop = True
    ),
    action_dim=action_dim,
    action_space_high=action_space_high, 
    action_space_low=action_space_low, 
    use_continual_backprop = True, 
    laser_dim = laser_dim,
    input_dim = input_dim,
)

_, log_probs, _ = actor.get_action_alt(state)

input_dim_critic = input_dim + action_dim
critic = CrossQCritic(
        state_preprocess=TCNEncoder((4, laser_dim), 2, 512, use_continual_backprop = True, batch_norm = True),
        head=MLP_CrossQ(
            input_dim_critic,
            2,
            512,
            use_continual_backprop = True
        ), use_continual_backprop = True,
        laser_dim = laser_dim,
        )

gamma = 0.99
rewards_scale = 1.0
init_temperature = 1.0
log_alpha = torch.tensor(
    [np.log(init_temperature)],
    requires_grad=True,
    dtype=torch.float32,
)

for i in range(200):  # Train the actor and critic twice
    # Generate random state and action data
    random_state = torch.rand_like(state)
    random_action = torch.rand_like(action)

    # Forward pass through the critic
    cat_state = torch.cat([random_state, random_state], dim=0)
    cat_actions = torch.cat([random_action, random_action], dim=0)
    cat_q1, cat_q2 = critic(cat_state, cat_actions)

    q_values_1, q_values_1_next = torch.chunk(cat_q1, chunks=2, dim=0)
    q_values_2, q_values_2_next = torch.chunk(cat_q2, chunks=2, dim=0)

    # Compute target Q values
    _, log_probs, _ = actor.get_action_alt(random_state)
    target_q_values = (
        torch.minimum(q_values_1_next, q_values_2_next)
        - log_alpha.exp() * log_probs
    )

    reward = torch.rand([2, 1])  # Random rewards
    termination = torch.randint(0, 2, [2, 1]).float()  # Random termination flags
    q_target = (
        reward * rewards_scale
        + gamma * (1 - termination) * target_q_values
    ).detach()

    # Compute losses
    q1_loss = F.mse_loss(q_values_1, q_target)
    q2_loss = F.mse_loss(q_values_2, q_target)
    total_q_loss = q1_loss + q2_loss

    actor_loss = (log_alpha.exp() * log_probs - target_q_values).mean()

    # Backpropagation
    #total_q_loss.backward()
    actor_loss.backward(retain_graph=True)

    print(f"Iteration {i + 1}:")
    print(f"Q1 Loss: {q1_loss.item()}, Q2 Loss: {q2_loss.item()}, Actor Loss: {actor_loss.item()}")


Action scale:  tensor([1.5000, 3.1400]) Action bias:  tensor([0.5000, 0.0000])
Iteration 1:
Q1 Loss: 42.859588623046875, Q2 Loss: 37.85688400268555, Actor Loss: 31.03331756591797
Iteration 2:
Q1 Loss: 0.3729541599750519, Q2 Loss: 0.5197122693061829, Actor Loss: 34.232078552246094
Iteration 3:
Q1 Loss: 0.6571318507194519, Q2 Loss: 0.40473511815071106, Actor Loss: 29.983013153076172
Iteration 4:
Q1 Loss: 1.1364370584487915, Q2 Loss: 0.20678766071796417, Actor Loss: 25.47156524658203
Iteration 5:
Q1 Loss: 0.8213761448860168, Q2 Loss: 0.24949470162391663, Actor Loss: 29.63248634338379
Iteration 6:
Q1 Loss: 0.23570895195007324, Q2 Loss: 0.2561352550983429, Actor Loss: 33.74494171142578
Iteration 7:
Q1 Loss: 0.20752078294754028, Q2 Loss: 0.17300297319889069, Actor Loss: 31.9905948638916
Iteration 8:
Q1 Loss: 90.01509857177734, Q2 Loss: 86.22649383544922, Actor Loss: 29.019248962402344
Iteration 9:
Q1 Loss: 263.6072082519531, Q2 Loss: 293.6383361816406, Actor Loss: 32.95869064331055
Iteration

In [ ]:
# cat_state = torch.cat([state, state], dim=0)
# cat_actions = torch.cat([action, action], dim=0)
# cat_q1, cat_q2 = critic(cat_state, cat_actions)

# q_values_1, q_values_1_next = torch.chunk(cat_q1, chunks=2, dim=0)
# q_values_2, q_values_2_next = torch.chunk(cat_q2, chunks=2, dim=0)

# # print(q_values_1)
# # print(q_values_1.detach().mean())
# # print(q_values_2)

# init_temperature = 1.0

# log_alpha = torch.tensor(
#     [np.log(init_temperature)],
#     requires_grad=True,
#     dtype=torch.float32,
# )

# target_q_values = (
#     torch.minimum(q_values_1_next, q_values_2_next)
#     - log_alpha.exp() * log_probs
# )

# print("q_values_1 shape:", q_values_1.shape)
# print("q_values_2 shape:", q_values_2.shape)
# print("q_values_1_next shape:", q_values_1_next.shape)
# print("q_values_2_next shape:", q_values_2_next.shape)
# print("target_q_values shape:", target_q_values.shape)

# rewards_scale = 1.0
# reward = torch.ones([2, 1])
# termination = torch.zeros([2, 1])
# gamma = 0.99

# q_target = (
#     reward * rewards_scale
#     + gamma * (1 - termination) * target_q_values
# ).detach()

# print("q_target shape:", q_target.shape)

# q1_loss = F.mse_loss(q_values_1, q_target)
# q2_loss = F.mse_loss(q_values_2, q_target)
# total_q_loss = q1_loss + q2_loss

# actor_loss = (log_alpha.exp() * log_probs - target_q_values).mean()
# #actor_loss.backward()

# total_q_loss.backward()

In [ ]:
from sac.rl import CrossQ_SAC

from sac.net import TCNEncoder, MLP_CrossQ
from sac.rl import CrossQCritic, Actor
import torch
from gym.spaces import Box
import numpy as np

def get_encoder(encoder_type, args):
    if encoder_type == "tcn":
        return TCNEncoder(**args)
    else:
        raise 
    

min_v=-1
max_v=2
min_w=-3.14
max_w=3.14

range_dict = RANGE_DICT = {
            "linear_velocity": [min_v, max_v],
            "angular_velocity": [min_w, max_w],
        }

action_space = Box(
            low=np.array([RANGE_DICT["linear_velocity"][0], RANGE_DICT["angular_velocity"][0]]),
            high=np.array([RANGE_DICT["linear_velocity"][1], RANGE_DICT["angular_velocity"][1]]),
            dtype=np.float32
        )

device = "cuda:0" if torch.cuda.is_available() else "cpu"
state_dim = [2, 4, 726]
encoder_type = "tcn"
encoder_args = {
    "input_dim": state_dim,
    "num_layers": 2,
    "hidden_size": 512,
    "history_length": 4,
}

input_dim = 744 # Input dim is [laser dimensio + stack frames*size of local goal + stack frames*action dim] + local_goal * stack frames
action_dim = 2
action_space_high = action_space.high
actor = Actor(
    state_preprocess=get_encoder(encoder_type, encoder_args),
    head=MLP_CrossQ(
        input_dim,
        2,
        512,
    ),
    action_dim=action_dim,
    input_dim=input_dim,
    action_space_high=action_space.high, 
    action_space_low=action_space.low
).to(device)

print("Total number of parameters: %d" % sum(p.numel() for p in actor.parameters()))
input_dim += np.prod(action_dim)

critic = CrossQCritic(
    state_preprocess=get_encoder(encoder_type, encoder_args),
    head=MLP_CrossQ(
        input_dim,
        2,
        512,
    ),
).to(device)

critic_optim = torch.optim.Adam(
    critic.parameters(), lr=1e-5
)
actor_optim = torch.optim.Adam(actor.parameters(), lr=2e-5)
print(device)
policy = CrossQ_SAC(
    actor=actor,
    actor_optim=actor_optim,
    critic=critic,
    critic_optim=critic_optim,
    action_range=[action_space.low, action_space.high],
    device=device,  # TODO: review this
)

policy.save("funny_run")

Total number of parameters: 3546588
cuda:0


/home/bbruno/anaconda3/envs/rl_env/lib/python3.13/site-packages/gym/spaces/box.py:127: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")
/home/bbruno/Documents/the-barn-challenge-CrossQ/end_to_end/sac/rl.py:47: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /home/conda/feedstock_root/build_artifacts/libtorch_1739249442529/work/torch/csrc/utils/tensor_new.cpp:278.)
  self.target_entropy = -torch.prod(torch.Tensor(action_range)).to(self.device)


PermissionError: [Errno 13] Permission denied: '/root/e2e_crossq'

In [ ]:
from sac.net import MLP_CrossQ
import torch
from torch import nn

input_dim = 15

my_net = MLP_CrossQ(input_dim=input_dim)
print("Layers:", my_net.mlp)
input = torch.ones(2, 15)

# Forward pass
output = my_net(input)

# Define a dummy loss (mean of the output)
loss = output.mean()

# Backward pass
loss.backward()


Layers: Sequential(
  (0): BatchRenorm()
  (1): Linear(in_features=15, out_features=512, bias=True)
  (2): ReLU6()
  (3): BatchRenorm()
  (4): CBPLinear(
    (in_layer): Linear(in_features=15, out_features=512, bias=True)
    (out_layer): Linear(in_features=512, out_features=512, bias=True)
    (bn_layer): BatchRenorm()
  )
  (5): Linear(in_features=512, out_features=512, bias=True)
  (6): ReLU6()
  (7): BatchRenorm()
)


In [ ]:
from sac.net import MLP_CrossQ, TCNEncoder
from sac.rl import CrossQCritic, Actor
import torch
from torch import nn

input_dim = 15

my_net = MLP_CrossQ(input_dim=input_dim)
print("Layers:", my_net.mlp)
input = torch.ones(2, 15)

# Forward pass
output = my_net(input)

# Define a dummy loss (mean of the output)
loss = output.mean()

# Backward pass
loss.backward()